In [1]:
import os
import pandas as pd
import boto3
from tqdm import tqdm
import datetime as dt

In [2]:
print(f'Last run date: {dt.datetime.today()}')

Last run date: 2024-07-01 16:30:14.241536


#### constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

Project: 20240423-gen-xii-payload-parsing
Task: 04_concat_results


#### get names of files in s3 bucket 

In [5]:
# init
cls_client = boto3.client('s3')

# prefix
str_prefix = '03_step_function/parsed'

# list the files
dict_response = cls_client.list_objects_v2(Bucket=str_project, Prefix=str_prefix)

# get contents
list_dict_contents = dict_response['Contents']

# get the filenames
list_str_files = [dict_contents['Key'] for dict_contents in list_dict_contents]

# get only gzip
list_str_files = [str_file for str_file in list_str_files if '.gzip' in str_file]
print(f'There are {len(list_str_files)} parsed files:')
for a, str_file in enumerate(list_str_files):
    print(f'{a+1} - {str_file}')

There are 230 parsed files:
1 - 03_step_function/parsed/X_raw_1.gzip
2 - 03_step_function/parsed/X_raw_10.gzip
3 - 03_step_function/parsed/X_raw_100.gzip
4 - 03_step_function/parsed/X_raw_101.gzip
5 - 03_step_function/parsed/X_raw_102.gzip
6 - 03_step_function/parsed/X_raw_103.gzip
7 - 03_step_function/parsed/X_raw_104.gzip
8 - 03_step_function/parsed/X_raw_105.gzip
9 - 03_step_function/parsed/X_raw_106.gzip
10 - 03_step_function/parsed/X_raw_107.gzip
11 - 03_step_function/parsed/X_raw_108.gzip
12 - 03_step_function/parsed/X_raw_109.gzip
13 - 03_step_function/parsed/X_raw_11.gzip
14 - 03_step_function/parsed/X_raw_110.gzip
15 - 03_step_function/parsed/X_raw_111.gzip
16 - 03_step_function/parsed/X_raw_112.gzip
17 - 03_step_function/parsed/X_raw_113.gzip
18 - 03_step_function/parsed/X_raw_114.gzip
19 - 03_step_function/parsed/X_raw_115.gzip
20 - 03_step_function/parsed/X_raw_116.gzip
21 - 03_step_function/parsed/X_raw_117.gzip
22 - 03_step_function/parsed/X_raw_118.gzip
23 - 03_step_func

#### Concatenate files

In [6]:
%%time

list_df = []
for str_file in tqdm(list_str_files):
    # read
    str_uri = f's3://{str_project}/{str_file}'
    df = pd.read_parquet(str_uri)
    # append
    list_df.append(df)
# concat
df = pd.concat(list_df)
# save memory
del list_df

# show
df

100%|██████████| 230/230 [01:29<00:00,  2.56it/s]


CPU times: user 1min 4s, sys: 18.9 s, total: 1min 23s
Wall time: 1min 35s


,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,bigstatusid__app,strcity__app,strname__app,strzipcode__app,applicationdate__app,bitapproved__app,...,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target,bigAccountId,dtmFunded,BITDEBTOR,tu_was_empty__tu
6507952__primary__20221222,0__0__20221222,6507952,NaN,1,11,WAKARUSA,Indiana,46573,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,6507952,2023-01-03,1,NaN
6514953__primary__20221227,0__0__20221227,6514953,NaN,1,11,PHOENIX,Arizona,85016,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,6514953,2023-01-05,1,NaN
6523250__primary__20221231,0__0__20221231,6523250,NaN,1,11,CHICAGO,Illinois,60805,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,6523250,2023-01-10,1,NaN
6505279__primary__20221220,0__0__20221220,6505279,NaN,1,11,Slippery Rock,Pennsylvania,16057,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,6505279,2023-01-12,1,NaN
6491983__primary__20221213,0__0__20221213,6491983,NaN,1,11,DETROIT,Michigan,48213,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,6491983,2023-01-17,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7351323__primary__20231116,0__0__20231116,7351323,NaN,1,9,Burke,Virginia,22015,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,7351323,2023-11-30,1,NaN
7357885__primary__20231118,0__0__20231118,7357885,NaN,1,5,San Diego,California,92119,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,7357885,2023-12-04,1,NaN
7365253__primary__20231122,0__0__20231122,7365253,NaN,1,1,CUMBERLAND,Maryland,21502,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,7365253,2023-12-08,1,NaN
7333795__primary__20231114,0__0__20231114,7333795,NaN,1,1,BRAYMER,Missouri,64624,NaN,False,...,NaN,NaN,NaN,NaN,NaN,NaN,7333795,2023-12-20,1,NaN


In [7]:
list_cols_ = [col for col in list(df.columns) if '__ln' in col]
list_cols_

['uniqueid__ln',
 'biglnriskviewattributesv5id__ln',
 'bigaccountid__ln',
 'bigdebtorid__ln',
 'biglnriskviewscoreid__ln',
 'bitinvalid__ln',
 'dtmstampcreation__ln',
 'attribute_index__ln',
 'inputprovidedfirstname__ln',
 'inputprovidedlastname__ln',
 'inputprovidedstreetaddress__ln',
 'inputprovidedcity__ln',
 'inputprovidedstate__ln',
 'inputprovidedzipcode__ln',
 'inputprovidedssn__ln',
 'inputprovideddateofbirth__ln',
 'inputprovidedphone__ln',
 'inputprovidedlexid__ln',
 'subjectrecordtimeoldest__ln',
 'subjectrecordtimenewest__ln',
 'subjectnewestrecord12month__ln',
 'subjectactivityindex03month__ln',
 'subjectactivityindex06month__ln',
 'subjectactivityindex12month__ln',
 'subjectage__ln',
 'subjectdeceased__ln',
 'subjectssncount__ln',
 'subjectstabilityindex__ln',
 'subjectstabilityprimaryfactor__ln',
 'subjectabilityindex__ln',
 'subjectabilityprimaryfactor__ln',
 'subjectwillingnessindex__ln',
 'subjectwillingnessprimaryfactor__ln',
 'confirmationsubjectfound__ln',
 'confir

In [8]:
df_tmp = df[list_cols_]
df_tmp.head(10)

,uniqueid__ln,biglnriskviewattributesv5id__ln,bigaccountid__ln,bigdebtorid__ln,biglnriskviewscoreid__ln,bitinvalid__ln,dtmstampcreation__ln,attribute_index__ln,inputprovidedfirstname__ln,inputprovidedlastname__ln,...,alert2__ln,alert3__ln,alert4__ln,alert5__ln,alert6__ln,alert7__ln,alert8__ln,alert9__ln,alert10__ln,intscore__ln
6507952__primary__20221222,"6507952,8129537",0,6507952,8129537,0,False,2022-12-22T10:08:57,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6514953__primary__20221227,"6514953,8138048",0,6514953,8138048,0,False,2022-12-27T19:12:51,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6523250__primary__20221231,"6523250,8148207",0,6523250,8148207,0,False,2022-12-31T14:50:24,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6505279__primary__20221220,"6505279,8126294",0,6505279,8126294,0,False,2022-12-20T15:03:08,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6491983__primary__20221213,"6491983,8109984",0,6491983,8109984,0,False,2022-12-13T10:45:17,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6551440__primary__20230116,"6551440,8182749",0,6551440,8182749,0,False,2023-01-16T13:57:19,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6527403__primary__20230103,"6527403,8153295",0,6527403,8153295,0,False,2023-01-03T17:30:56,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6561720__primary__20230121,"6561720,8195257",0,6561720,8195257,0,False,2023-01-21T10:58:27,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6561546__primary__20230121,"6561546,8195030",0,6561546,8195030,0,False,2023-01-21T10:30:47,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6551216__primary__20230116,"6551216,8182472",0,6551216,8182472,0,False,2023-01-16T12:59:53,NaN,1.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.purchaseactivityindex__ln.value_counts()

purchaseactivityindex__ln
 0.0    26954
-1.0       73
Name: count, dtype: int64

#### Write to s3

In [10]:
%%time

# convert to string
for col in df.columns:
    if df[col].dtype not in ['int64', 'float64']:
        df[col] = df[col].astype(str)

str_filename = 'df_X_raw.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 8.6 s, sys: 25.3 ms, total: 8.62 s
Wall time: 8.71 s
